# 16 - Analogo en R (recreado desde cero)

## Tema base

**16 - Decoradores: extender comportamiento sin tocar la funcion original**


## Objetivos de la sesion en R

1. Mantener la teoria clave del notebook Python en equivalente R.
2. Usar solo codigo ejecutable en R.
3. Conservar ejercicios adaptados a R.
4. Agregar probabilidad y estadistica acorde al tema.


## Ruta Teoria-Codigo (alternada)

Se organiza la sesion en bloques cortos que alternan concepto y practica.


### Bloque teorico 1

### Teoria 1

# 16 - Decoradores: extender comportamiento sin tocar la funcion original

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender que problema resuelven los decoradores.
2. Explicar la relacion entre funciones, closures y decoradores.
3. Construir decoradores basicos y robustos.
4. Usar `@decorador` entendiendo que hace R por debajo.
5. Preservar metadatos con `functools.wraps`.
6. Crear decoradores con parametros (fabricas de decoradores).
7. Combinar decoradores y razonar sobre su orden de aplicacion.
8. Identificar errores sutiles (retornos perdidos, estado compartido, efectos colaterales).
9. Aplicar decoradores en casos reales: logging, validacion, cache y control de acceso.
10. Tomar decisiones de diseno: cuando conviene un decorador y cuando no.

### Teoria 2

## 1. Problema que resuelven los decoradores

A veces quieres agregar capacidades transversales a varias funciones:
- medir tiempo,
- registrar logs,
- validar argumentos,
- controlar permisos,
- cachear resultados.

Sin decoradores, repites codigo alrededor de la logica principal.
Con decoradores, encapsulas ese comportamiento extra en una sola pieza reutilizable.

### Teoria 3

## 2. Base conceptual: funciones como objetos + closures

Un decorador es posible porque en R:
1. Las funciones son objetos de primera clase.
2. Una funcion puede devolver otra funcion.
3. Las funciones internas pueden recordar variables del contexto externo (closure).

Sin esta base, decorar no tendria sentido.

### Teoria 4

## 3. Primer decorador manual

Un decorador recibe una funcion y devuelve otra funcion.
La funcion retornada (wrapper) llama a la original y agrega comportamiento.

### Teoria 5

## 4. Azucar sintactica: `@decorador`

Estas dos formas son equivalentes:

1. `f = decorador(f)`
2. usar `@decorador` justo encima de `function f(...):`

`@decorador` mejora legibilidad y deja clara la intencion.

### Teoria 6

## 5. Decoradores generales con `*args` y `**kwargs`

Error comun: crear wrappers que solo funcionan sin parametros.
Para ser reutilizable, el wrapper debe aceptar cualquier firma.


In [ ]:
medir_tiempo <- function(fun){force(fun); function(...){t0 <- Sys.time(); out <- fun(...); t1 <- Sys.time(); cat('Tiempo:', round(as.numeric(t1-t0, units='secs'),6), 's\n'); out}}
suma_lenta <- function(n){total <- 0; for(i in 1:n) total <- total+i; total}
suma_t <- medir_tiempo(suma_lenta)
suma_t(1e6)


### Bloque teorico 2

### Teoria 7

## 6. Preservar metadatos con `functools.wraps`

Sin `@wraps`, la funcion decorada pierde datos importantes:
- `__name__`
- `__doc__`
- otras propiedades de introspeccion

Esto afecta debugging, ayuda interactiva y herramientas automaticas.

### Teoria 8

## 7. Decoradores con parametros (fabrica de decoradores)

Cuando quieres configurar el decorador (por ejemplo nivel de log),
necesitas una funcion externa adicional.

Estructura tipica:
1. fabrica(parametros)
2. decorador(func)
3. wrapper(*args, **kwargs)

### Teoria 9

## 8. Caso real 1: medicion de tiempo

Medir tiempos alrededor de funciones es un caso clasico.
`time.perf_counter()` es adecuado para medir intervalos cortos.

### Teoria 10

## 9. Caso real 2: validacion de argumentos

Un decorador puede verificar precondiciones antes de ejecutar logica principal.
Si la validacion falla, lanza excepcion con mensaje claro.

### Teoria 11

## 10. Caso real 3: cache simple

Si una funcion pura recibe los mismos argumentos varias veces,
puedes guardar resultados para evitar recalculo.

Advertencia:
- Este ejemplo asume argumentos hashables.
- Para casos productivos, revisa `functools.lru_cache`.

### Teoria 12

## 11. Orden de aplicacion cuando apilas decoradores

Si tienes:

(Ejemplo de codigo trasladado a celdas R equivalentes.)

R interpreta: `f = A(B(f))`

Eso significa:
- `B` se aplica primero.
- luego `A` envuelve el resultado.

Razonar este orden evita bugs al combinar validacion, cache y logging.


In [ ]:
con_log <- function(fun){force(fun); function(...){cat('[LOG] inicio\n'); out <- fun(...); cat('[LOG] fin\n'); out}}
raiz_log <- con_log(sqrt)
raiz_log(144)


### Bloque teorico 3

### Teoria 13

## 12. Decoradores y manejo de errores

Un decorador puede transformar excepciones o aplicar reintentos.
Pero debes evitar ocultar errores de forma silenciosa.

Regla util: si capturas una excepcion, decide explicitamente si:
- la vuelves a lanzar,
- devuelves un valor seguro,
- o la conviertes en otra mas semantica.

### Teoria 14

## 13. Decoradores basados en clase

No siempre un decorador debe ser funcion.
Una clase con `__call__` tambien puede actuar como decorador.

Ventaja: encapsular estado y comportamiento adicional de forma explicita.

### Teoria 15

## 14. Buenas practicas

1. Usa `@wraps` casi siempre.
2. Mantiene los decoradores pequenos y con una sola responsabilidad.
3. No ocultes errores importantes sin justificacion.
4. Documenta efectos secundarios (logs, cache, retries, permisos).
5. Evita decoradores "magicos" que dificulten leer el flujo.
6. Si la logica extra es muy especifica y local, una llamada explicita puede ser mejor que decorar.

### Teoria 16

## 15. Errores comunes

1. Olvidar `return func(...)` en el wrapper.
2. No aceptar `*args, **kwargs` y romper funciones con firma distinta.
3. Compartir estado mutable entre funciones sin querer.
4. Aplicar decoradores en orden incorrecto.
5. Usar cache sobre funciones con efectos secundarios.
6. Decorar demasiado y volver opaco el comportamiento real.

### Teoria 17

## 17. Resumen de conceptos clave

1. Un decorador envuelve una funcion para extender comportamiento.
2. `@decorador` es sintaxis para reasignar funciones.
3. `*args`, `**kwargs` y `@wraps` son piezas fundamentales.
4. El orden de decoradores cambia el comportamiento final.
5. Decorar bien mejora reutilizacion; decorar mal oculta logica.


In [ ]:
suma_log_t <- con_log(medir_tiempo(suma_lenta))
suma_log_t(5e5)


## Extra de probabilidad y estadistica

Wrapper de Monte Carlo para resumen con intervalo empirico.


In [ ]:
repetir_mc <- function(fun, reps=1000){v <- replicate(reps, fun()); c(media=mean(v), q025=quantile(v,0.025), q975=quantile(v,0.975))}
set.seed(99)
repetir_mc(function() mean(rnorm(30)), 2000)


## Analogias R y Python

- Decoradores Python y wrappers R son el mismo patron de extension.
- Timing/logging se aplica igual en ambos.


In [ ]:
# Checklist rapido R vs Python
# 1) ?Que cambia en sintaxis?
# 2) ?Que cambia en estructura de datos?
# 3) ?Que cambia en manejo de NA/errores?


## Errores comunes

- Encadenar wrappers sin contrato claro.
- Cambiar firmas de funcion accidentalmente.
- Medir tiempo con muestras no comparables.


In [ ]:
# Auto-verificacion de errores comunes
# Define un caso borde y valida que tu solucion en R no falle silenciosamente.


## Ejercicios para pensar (no copiar codigo)

Primero argumenta en texto. Luego, si hace falta, valida con experimentos en R.


### Ejercicio reflexivo 1

### Ejercicio 1

## 16. Ejercicios de pensamiento cuidadoso

Estos ejercicios apuntan a razonamiento tecnico, no solo sintaxis:
- composicion,
- flujo de ejecucion,
- diseno,
- y efectos secundarios.


### Ejercicio reflexivo 2

### Ejercicio 2

### Ejercicio 1: refactor de repeticion transversal

**Tarea**: Tienes varias funciones con el mismo bloque de logging antes/despues.
Crea un decorador `loggear` y elimina repeticion sin cambiar el resultado final.

Objetivo: distinguir entre logica de negocio y logica transversal.


### Ejercicio reflexivo 3

### Ejercicio 3

### Ejercicio 2: bug silencioso por retorno perdido

**Tarea**: Este decorador imprime mensajes, pero la funcion decorada siempre retorna `None`.
Encuentra el error y corrigelo.


### Ejercicio reflexivo 4

### Ejercicio 4

### Ejercicio 3: preservacion de metadatos

**Tarea**: Decora una funcion documentada y compara `__name__` y `__doc__`
antes y despues de usar `functools.wraps`.

Explica por que importa en proyectos reales.


### Ejercicio reflexivo 5

### Ejercicio 5

### Ejercicio 4: decorador parametrizable de permisos

**Tarea**: Implementa `requiere_rol(rol_minimo)`.
La funcion decorada recibira `usuario` (dict con campo `rol`).
Si no cumple el rol, debe lanzar `PermissionError`.

Piensa como representar jerarquia de roles de forma limpia.


### Ejercicio reflexivo 6

### Ejercicio 6

### Ejercicio 5: razonar orden de decoradores

**Tarea**: Predice la salida exacta sin ejecutar. Luego verifica.
Finalmente, invierte el orden y explica la diferencia.


### Ejercicio reflexivo 7

### Ejercicio 7

### Ejercicio 6: reintentos con limites claros

**Tarea**: Implementa `reintentar(max_intentos, excepciones)`.
Debe reintentar solo para las excepciones indicadas y relanzar al agotar intentos.

Pregunta de diseno: conviene dormir entre intentos? quien define ese tiempo?


### Ejercicio reflexivo 8

### Ejercicio 8

### Ejercicio 7: cache y argumentos mutables

**Tarea**: Este enfoque falla con listas por no ser hashables.
Propone dos soluciones y analiza tradeoffs.

1. Convertir entradas mutables a inmutables.
2. Restringir API a argumentos hashables.


### Ejercicio reflexivo 9

### Ejercicio 9

### Ejercicio 8: rate limit testeable

**Tarea**: Crea `limitar_llamadas(max_llamadas, ventana_segundos, reloj)`.
`reloj` sera una funcion inyectada para facilitar pruebas.
Si se excede el limite, lanzar `RuntimeError`.

Clave del ejercicio: diseno para testabilidad.


### Ejercicio reflexivo 10

### Ejercicio 10

### Ejercicio 9: decorador de clase con estadisticas

**Tarea**: Implementa un decorador basado en clase `MetricasLlamada`
que registre:
1. total de llamadas,
2. ultima duracion,
3. promedio acumulado.

Debe funcionar con cualquier firma de funcion.


### Ejercicio reflexivo 11

### Ejercicio 11

### Ejercicio 10: criterio de arquitectura

**Tarea**: Para cada caso, decide si usar decorador o llamada explicita y justifica:
1. validar formato de email en 2 funciones.
2. abrir/cerrar transaccion en 30 operaciones de BD.
3. transformar un resultado solo en un endpoint unico.
4. registrar auditoria legal obligatoria en multiples acciones criticas.

No hay una unica respuesta correcta; evalua claridad, trazabilidad y mantenimiento.


### Ejercicio reflexivo 12

### Ejercicio 12

## 18. Reto final opcional

Disena un mini sistema de tareas con tres decoradores combinables:
1. `@requiere_rol(...)`
2. `@medir_tiempo`
3. `@auditar(evento=...)`

Objetivo: razonar orden, errores, trazabilidad y legibilidad del flujo.


### Preguntas de cierre

1. ?Que supuesto estadistico es mas fragil en esta clase?
2. ?Que evidencia minima pedirias antes de usar este enfoque en un caso real?
3. ?Como explicarias este tema a alguien no tecnico sin perder rigor?
4. ?Que cambiaria entre una implementacion en R y una en Python para produccion?
